# GEO real-brand pilot replay
Five measured consumer answers, not a new engine run. Run All: standard library only, no downloads or API calls. Output: audit-output/audit.md.
For fresh answers, use the embedded script in collect mode as documented in geo/README.md.

In [ ]:
from pathlib import Path
import subprocess, sys
Path('geo_audit.py').write_text('#!/usr/bin/env python3\n"""Frozen GEO audit procedure. Python standard library only; no provider SDK.\n\nConsumer mode prints prompts and accepts verbatim captures. Gemini API mode is\nexplicit opt-in for a billing-disabled project; never impersonates consumer UI.\nAnalyze, report and manifest operations are offline and deterministic.\n"""\nimport argparse\nfrom collections import Counter\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport sys\nimport urllib.error\nimport urllib.parse\nimport urllib.request\nimport uuid\n\nVERSION = "geo-audit-1"\nENGINES = {"chatgpt-consumer", "perplexity-consumer", "gemini-consumer", "gemini-api"}\nSTATUSES = {"ok", "blocked", "error", "unavailable"}\n\n\ndef digest(text):\n    return hashlib.sha256(text.encode("utf-8")).hexdigest()\n\n\ndef stable(value):\n    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)\n\n\ndef validate_config(c):\n    assert c["brand"] in [b["name"] for b in c["brands"]], "target brand missing"\n    for key, rows in [("name", c["brands"]), ("id", c["queries"])]:\n        assert len({r[key] for r in rows}) == len(rows), "duplicate config identifiers"\n    assert c["queries"] and c["engines"] and set(c["engines"]) <= ENGINES\n    for q in c["queries"]:\n        assert q["prompt"].strip() and q["fixability_reason"].strip()\n        for k in ("intent", "fixability"):\n            assert type(q[k]) in (int, float) and math.isfinite(q[k]) and 0 <= q[k] <= 1\n    for b in c["brands"]:\n        assert b["aliases"] and all(a.strip() for a in b["aliases"])\n        assert b["domains"] and all(re.fullmatch(r"[a-z0-9.-]+", d) for d in b["domains"])\n    return c\n\n\ndef canonical_url(url):\n    p = urllib.parse.urlsplit(url)\n    if p.scheme not in ("http", "https") or not p.hostname or p.username or p.password:\n        raise ValueError("citation must be a public HTTP(S) URL without credentials")\n    host = p.hostname.lower().removeprefix("www.")\n    query = urllib.parse.urlencode([(k, v) for k, v in urllib.parse.parse_qsl(p.query)\n                                  if not k.lower().startswith("utm_")])\n    return urllib.parse.urlunsplit((p.scheme, host, p.path or "/", query, ""))\n\n\ndef belongs(url, brand):\n    host = urllib.parse.urlsplit(url).hostname or ""\n    return any(host == d or host.endswith("." + d) for d in brand["domains"])\n\n\ndef answer_text(record):\n    text = record["raw_answer"]\n    if record["format"] == "dom-snapshot":\n        # Source chips/drawers are citation evidence, not prose mentions.\n        lines, skip_depth = [], None\n        for line in text.splitlines():\n            depth = len(line)-len(line.lstrip())\n            if skip_depth is not None and depth > skip_depth:\n                continue\n            skip_depth = None\n            if line.lstrip().startswith((\'- dialog:\', \'- button "View source\', \'- link \')):\n                skip_depth = depth\n                continue\n            if not line.lstrip().startswith(\'- /url:\'):\n                lines.append(line)\n        return \'\\n\'.join(lines)\n    # A domain in a Markdown URL is not a textual brand mention.\n    text = re.sub(r"\\[([^\\]]+)\\]\\(https?://[^)]+\\)", r"\\1", text)\n    return re.sub(r"https?://\\S+", "", text)\n\n\ndef mentions(text, brand):\n    return any(re.search(r"(?<!\\w)" + re.escape(a) + r"(?!\\w)", text, re.I)\n               for a in brand["aliases"])\n\n\ndef wilson(k, n):\n    if not n:\n        return None\n    z = 1.959963984540054\n    p = k / n\n    den = 1 + z*z/n\n    mid = (p + z*z/(2*n)) / den\n    half = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / den\n    return [max(0, mid-half), min(1, mid+half)]\n\n\ndef opportunity(data):\n    n = data[\'n\']\n    if n == 0:\n        return None\n    return 100 * data[\'intent\'] * data[\'fixability\'] * (n - data[\'mentions\'] + 1) / (n + 2)\n\n\ndef read_records(path, config):\n    qs = {q["id"]: q for q in config["queries"]}\n    records, seen, exclusions = [], {}, []\n    for line_no, line in enumerate(Path(path).read_text().splitlines(), 1):\n        if not line.strip():\n            continue\n        r = json.loads(line)\n        for k in ("id", "query_id", "prompt", "engine", "model", "observed_at", "status",\n                  "evidence_kind", "raw_answer", "sha256", "format", "context", "citation_coverage", "citations"):\n            if k not in r:\n                raise ValueError(f"line {line_no}: missing {k}")\n        if r["evidence_kind"] != "measured":\n            exclusions.append({"id": r["id"], "reason": "illustrative/synthetic: excluded"})\n            continue\n        if r["engine"] not in ENGINES or r["status"] not in STATUSES:\n            raise ValueError("unknown engine or status")\n        if r["query_id"] not in qs or r["prompt"] != qs[r["query_id"]]["prompt"]:\n            raise ValueError("query text mismatch: make a new query id for a changed prompt")\n        if r["sha256"] != digest(r["raw_answer"]):\n            raise ValueError("raw evidence hash mismatch")\n        stamp = datetime.fromisoformat(r["observed_at"].replace("Z", "+00:00"))\n        if stamp.tzinfo is None:\n            raise ValueError("capture time requires timezone")\n        if not r["id"] or not r["model"] or not isinstance(r["context"], dict):\n            raise ValueError("capture identity/model/context required")\n        for k in ("session", "locale", "region", "search", "collector", "new_chat"):\n            if k not in r["context"]:\n                raise ValueError("missing collection context: " + k)\n        if r["format"] not in ("text", "dom-snapshot") or r["citation_coverage"] not in ("complete", "visible-only", "unknown"):\n            raise ValueError("unsupported capture format or citation coverage")\n        if r["status"] == "ok" and not r["raw_answer"].strip():\n            raise ValueError("empty answer cannot be a successful measurement")\n        for cite in r["citations"]:\n            canonical_url(cite["url"])\n            if cite["url"] not in r["raw_answer"]:\n                raise ValueError("citation URL missing from raw capture")\n        for brand, review in r.get(\'sentiment_reviews\', {}).items():\n            if brand not in [b[\'name\'] for b in config[\'brands\']] or review.get(\'label\') not in (\'positive\',\'negative\',\'mixed\',\'neutral\'):\n                raise ValueError(\'invalid sentiment annotation\')\n            if not review.get(\'quote\') or review[\'quote\'] not in r[\'raw_answer\'] or not review.get(\'reviewer\'):\n                raise ValueError(\'sentiment annotation needs exact evidence and reviewer\')\n        identity = stable(r)\n        if r["id"] in seen:\n            if seen[r["id"]] != identity:\n                raise ValueError("conflicting duplicate capture id")\n            exclusions.append({"id": r["id"], "reason": "duplicate capture id"})\n            continue\n        seen[r["id"]] = identity\n        records.append(r)\n    return records, exclusions\n\n\ndef stratum(r):\n    # Same-session repeats remain labelled; do not pretend they are independent users.\n    return stable({"engine": r["engine"], "model": r["model"], "context": r["context"],\n                   "citation_coverage": r["citation_coverage"]})\n\n\ndef analyze(config, records, exclusions=None):\n    brands = config["brands"]\n    target = next(b for b in brands if b["name"] == config["brand"])\n    rows, facts, sources = [], [], {}\n    measured = [r for r in records if r["evidence_kind"] == "measured"]\n    for r in measured:\n        if r["status"] != "ok":\n            continue\n        text = answer_text(r)\n        urls = sorted({canonical_url(c["url"]) for c in r["citations"]})\n        found = [b["name"] for b in brands if mentions(text, b)]\n        for name in found:\n            facts.append({"entity": name, "predicate": "mentioned", "status": "observed",\n                          "capture_id": r["id"], "sha256": r["sha256"], "query_id": r["query_id"]})\n        for url in urls:\n            owner = next((b["name"] for b in brands if belongs(url, b)), None)\n            s = sources.setdefault(url, {"url": url, "host": urllib.parse.urlsplit(url).hostname,\n                "publisher": owner, "kind": "brand-owned" if owner == target["name"] else "competitor-owned" if owner else "unclassified-third-party",\n                "capture_ids": [], "queries": [], "engines": [], "authority": None,\n                "authority_note": "Not independently assessed; citation frequency is not authority."})\n            s["capture_ids"].append(r["id"])\n            s["queries"] = sorted(set(s["queries"] + [r["query_id"]]))\n            s["engines"] = sorted(set(s["engines"] + [r["engine"]]))\n            facts.append({"entity": r["query_id"], "predicate": "answer_links_source", "value": url,\n                          "status": "observed", "capture_id": r["id"], "sha256": r["sha256"]})\n    for q in config["queries"]:\n        for engine in sorted(set(config["engines"]) | {r["engine"] for r in measured}):\n            candidates = [r for r in measured if r["query_id"] == q["id"] and r["engine"] == engine]\n            groups = sorted({stratum(r) for r in candidates}) or [None]\n            for group in groups:\n                attempts = [r for r in candidates if stratum(r) == group]\n                good = [r for r in attempts if r["status"] == "ok"]\n                n = len(good)\n                counts = {b["name"]: sum(mentions(answer_text(r), b) for r in good) for b in brands}\n                total = sum(counts.values())\n                metrics = {}\n                for b in brands:\n                    m = counts[b["name"]]\n                    citation_good = [r for r in good if r["citation_coverage"] != "unknown"]\n                    cited = sum(any(belongs(canonical_url(c["url"]), b) for c in r["citations"]) for r in citation_good)\n                    metrics[b["name"]] = {"mentions": m, "n": n, "visibility": m/n if n else None,\n                        "wilson95": wilson(m, n), "share_of_voice": m/total if total else None,\n                        "owned_domain_cited_answers": cited, "citation_n": len(citation_good),\n                        "owned_domain_citation_rate": cited/len(citation_good) if citation_good else None,\n                        "sentiment": dict(Counter(r[\'sentiment_reviews\'][b[\'name\']][\'label\'] for r in good if b[\'name\'] in r.get(\'sentiment_reviews\', {}))) or ("not-mentioned" if n and not m else "unassessed"),\n                        "sentiment_review_n": sum(b[\'name\'] in r.get(\'sentiment_reviews\',{}) for r in good),\n                        "sentiment_note": "Evidence-linked human/analyst annotations only; missing reviews are unassessed, not neutral."}\n                m = counts[target["name"]]\n                gap = (n-m+1)/(n+2) if n else None  # Beta(1,1) posterior mean absence\n                score = opportunity({\'n\':n, \'mentions\':m, \'intent\':q[\'intent\'], \'fixability\':q[\'fixability\']})\n                urls = sorted({canonical_url(c["url"]) for r in good for c in r["citations"]})\n                key = digest(stable([q["id"], engine, group]))[:16]\n                rows.append({"id": key, "query_id": q["id"], "prompt": q["prompt"], "engine": engine,\n                    "stratum": json.loads(group) if group else None, "n": n, "attempts": len(attempts),\n                    "failed": len(attempts)-n, "metrics": metrics, "intent_prior": q["intent"],\n                    "fixability_prior": q["fixability"], "fixability_reason": q["fixability_reason"],\n                    "opportunity_score": score, "score_kind": "prior-weighted hypothesis, not measured ROI",\n                    "absence_posterior_mean": gap, "evidence_ids": [r["id"] for r in good],\n                    "source_urls": urls, "framing": q["framing"],\n                    "status": "pilot-needs-replication" if n and n < 30 else "sampled" if n else "unmeasured"})\n    rows.sort(key=lambda r: (r["opportunity_score"] is None, -(r["opportunity_score"] or 0), r["id"]))\n    hypotheses = []\n    for row in rows:\n        if not row["n"]:\n            continue\n        third = [u for u in row["source_urls"] if not belongs(u, target)]\n        mechanisms = [("positioning", "Test a truthful use-case page against the exact buyer constraints.")]\n        if third:\n            mechanisms.append(("source-coverage", "Review these cited pages for factual omissions and eligibility; propose corrections only where supported."))\n        for mechanism, action in mechanisms:\n            hypotheses.append({"id": row["id"]+"-"+mechanism, "query_id": row["query_id"],\n                "engine": row["engine"], "mechanism": mechanism, "status": "untested-hypothesis",\n                "priority": row["opportunity_score"], "evidence_ids": row["evidence_ids"], "sources": third,\n                "action": action, "experiment": "Pre-register fixed prompts; collect fresh chats over multiple days before and after one factual change, with an unchanged comparison query. Log model/search conditions; report uncertainty. Do not infer causation from before/after alone."})\n    contrasts = []\n    for i, left in enumerate(rows):\n        for right in rows[i+1:]:\n            if not left[\'n\'] or not right[\'n\'] or left[\'stratum\'] != right[\'stratum\'] or left[\'query_id\'] == right[\'query_id\']:\n                continue\n            delta = right[\'metrics\'][target[\'name\']][\'visibility\'] - left[\'metrics\'][target[\'name\']][\'visibility\']\n            if abs(delta) >= 0.25:\n                contrasts.append({\'left\':left[\'query_id\'], \'right\':right[\'query_id\'], \'engine\':left[\'engine\'],\n                    \'observed_visibility_difference\':delta, \'sample_sizes\':[left[\'n\'],right[\'n\']],\n                    \'status\':\'descriptive-contrast-not-causal\', \'evidence_ids\':left[\'evidence_ids\']+right[\'evidence_ids\'],\n                    \'hypothesis\':\'Visibility may depend on buyer framing. Test a bridge from the visible use case to the missing category, without claiming unsupported product capabilities.\'})\n    return {"version": VERSION, "brand": config["brand"], "category": config["category"],\n        "methodology": config.get("methodology", "Operator supplied sampling design"),\n        "records": measured, "exclusions": exclusions or [], "opportunity_queue": rows,\n        "facts": facts, "sources": sorted(sources.values(), key=lambda s: (-len(s["capture_ids"]), s["url"])),\n        "hypotheses": hypotheses, "framing_contrasts": contrasts}\n\n\ndef pct(x):\n    return "unknown" if x is None else f"{100*x:.1f}%"\n\n\ndef md(text):\n    return str(text).replace("|", "\\\\|").replace("\\n", " ").replace("<", "&lt;")\n\n\ndef report(a):\n    ok = [r for r in a["records"] if r["status"] == "ok"]\n    lines = [f"# {md(a[\'brand\'])}: AI visibility pilot audit", "", "## Measured evidence", "",\n        f"{len(ok)} successful captured answers. This is a pilot, not a population visibility estimate.",\n        a["methodology"], "", "Only measured captures enter this report. No illustrative engine answers are included.",\n        "Models\' factual claims and linked sources have not been independently verified. Raw capture hashes establish integrity, not authenticity of operator-supplied imports.",\n        "", "| Query | Engine | Answers | Target mentions | Visibility | 95% Wilson interval* | Owned-domain link rate | Opportunity hypothesis /100 |",\n        "|---|---|---:|---:|---|---|---|---:|"]\n    for r in a["opportunity_queue"]:\n        m = r["metrics"][a["brand"]]\n        interval = "unknown" if m["wilson95"] is None else " to ".join(pct(x) for x in m["wilson95"])\n        score = "unknown" if r["opportunity_score"] is None else f"{r[\'opportunity_score\']:.1f}"\n        lines.append(f"| {md(r[\'query_id\'])} | {r[\'engine\']} | {r[\'n\']} | {m[\'mentions\'] if r[\'n\'] else \'unknown\'} | {pct(m[\'visibility\'])} | {interval} | {pct(m[\'owned_domain_citation_rate\'])} | {score} |")\n    lines += ["", "*Binomial interval assumes independent, identically distributed draws. Same-session repeats may be correlated; these intervals can understate uncertainty. Engine/model/context strata are separate rows. Failed requests never count as absence.",\n        "Visible-only citation captures provide lower bounds on links; unexpanded source drawers may contain more. An owned-domain link is not the same as a third-party citation supporting a claim about the brand.",\n        "", "## Competitor comparison", "", "Share of voice is binary brand-answer appearances divided by all tracked-brand appearances, within each stratum. Untracked brands are outside this denominator. Repeated mentions within one answer count once.", ""]\n    for r in a["opportunity_queue"]:\n        if not r["n"]:\n            continue\n        lines += [f"### {md(r[\'query_id\'])} / {r[\'engine\']} / {r[\'id\']}", "",\n                  "| Brand | Mentions / answers | Share of voice | Sentiment |", "|---|---:|---:|---|"]\n        for brand, m in r["metrics"].items():\n            lines.append(f"| {md(brand)} | {m[\'mentions\']}/{m[\'n\']} | {pct(m[\'share_of_voice\'])} | {m[\'sentiment\']} |")\n    lines += ["", "## Observed source map", "", "Citation frequency is observed; source authority is unknown. Competitor-owned comparison pages are explicitly labelled, not treated as independent authorities.", "",\n              "| Source | Publisher classification | Captured answers linking it | Authority |", "|---|---|---:|---|"]\n    for s in a["sources"]:\n        lines.append(f"| {md(s[\'url\'])} | {s[\'kind\']} | {len(s[\'capture_ids\'])} | unassessed |")\n    lines += ["", "## Proposed work, not measured results", "",\n        "Opportunity = 100 × declared buyer-intent prior × declared fixability prior × posterior mean absence (Beta(1,1) prior). No data means unknown, never zero visibility. Scores express investigation priority, not causal lift or revenue.", ""]\n    for c in a[\'framing_contrasts\']:\n        lines.append(f"- **Buyer-framing contrast:** {md(c[\'left\'])} vs {md(c[\'right\'])}, {c[\'engine\']}, sample sizes {c[\'sample_sizes\']}: observed difference {100*c[\'observed_visibility_difference\']:.1f} percentage points (right minus left). Descriptive only. {c[\'hypothesis\']}")\n    for r in a["opportunity_queue"]:\n        if r["n"]:\n            lines.append(f"- {md(r[\'query_id\'])}: intent prior {r[\'intent_prior\']}; fixability prior {r[\'fixability_prior\']}. {md(r[\'fixability_reason\'])}")\n    for h in a["hypotheses"][:6]:\n        lines += ["", f"- **{md(h[\'query_id\'])}: {h[\'mechanism\']} (untested).** {h[\'action\']} Evidence: {\', \'.join(h[\'evidence_ids\'])}. {h[\'experiment\']}"]\n    lines += ["", "## Evidence register and gaps", ""]\n    for r in a["records"]:\n        lines.append(f"- {r[\'id\']}: {r[\'engine\']}, {r[\'observed_at\']}, {r[\'status\']}, model: {md(r[\'model\'])}, SHA-256 `{r[\'sha256\']}`. Context: `{stable(r[\'context\'])}`.")\n    for r in a["opportunity_queue"]:\n        if not r["n"]:\n            lines.append(f"- Unmeasured: {r[\'engine\']} / {r[\'query_id\']}; collect an answer before assessing visibility.")\n    lines += ["", "Consumer captures require an accessible consumer session and operator assistance. The offline runner does not claim unattended access to ChatGPT, Perplexity or Gemini consumer products. Optional free-tier API output is a separate engine surface.",\n              "No verified uplift, revenue effect, sentiment classifier or independent source-authority measurement is claimed.", ""]\n    return "\\n".join(lines)\n\n\ndef make_record(q, engine, raw, status="ok", citations=None, model="not-disclosed", context=None):\n    return {"id": str(uuid.uuid4()), "query_id": q["id"], "prompt": q["prompt"], "engine": engine,\n        "model": model, "observed_at": datetime.now(timezone.utc).isoformat(), "status": status,\n        "evidence_kind": "measured", "raw_answer": raw, "sha256": digest(raw), "format": "text",\n        "context": context or {"session": "operator-declared", "locale": "unknown", "region": "unknown", "search": "unknown", "collector": "manual-paste", "new_chat": "operator-declared"},\n        "citation_coverage": "unknown", "citations": citations or []}\n\n\ndef collect(config, output, engine, repeats, api=False, model=None, billing_disabled=False):\n    if not 1 <= repeats <= 10:\n        raise ValueError("repeats must be 1..10 per invocation")\n    if api and (not billing_disabled or not model or not os.environ.get("GEMINI_API_KEY")):\n        raise ValueError("API requires --billing-disabled, --model and GEMINI_API_KEY; no paid fallback")\n    if api and not re.fullmatch(r"[a-zA-Z0-9.-]+", model):\n        raise ValueError("invalid model identifier")\n    Path(output).parent.mkdir(parents=True, exist_ok=True)\n    for q in config["queries"]:\n        for _ in range(repeats):\n            if api:\n                endpoint = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"\n                request = urllib.request.Request(endpoint, data=json.dumps({"contents": [{"parts": [{"text": q["prompt"]}]}]}).encode(),\n                    headers={"Content-Type": "application/json", "x-goog-api-key": os.environ["GEMINI_API_KEY"]})\n                try:\n                    with urllib.request.urlopen(request, timeout=60) as response:\n                        payload = json.loads(response.read(2_000_001))\n                    candidate = payload.get("candidates", [{}])[0]\n                    raw = "\\n".join(p.get("text", "") for p in candidate.get("content", {}).get("parts", []) if not p.get("thought"))\n                    status = "ok" if raw.strip() and candidate.get("finishReason") == "STOP" else "error"\n                    r = make_record(q, "gemini-api", raw or "No complete answer returned", status, model=payload.get("modelVersion", model))\n                    r["context"].update({"collector": "gemini-generateContent", "search": "off", "new_chat": True, "session": "stateless-api"})\n                    r["citation_coverage"] = "unknown"\n                    r["provider_response"] = payload\n                except (urllib.error.URLError, ValueError, TimeoutError) as exc:\n                    # Do not persist exception strings that might echo credentials.\n                    r = make_record(q, "gemini-api", f"Request failed: {type(exc).__name__}", "error", model=model)\n            else:\n                print(f"\\nOpen a FRESH chat in {engine}. Submit exactly:\\n{q[\'prompt\']}\\nPaste the answer (including visible URLs); finish with a line containing END. Type BLOCKED for an inaccessible engine.")\n                lines = []\n                while (line := input()) != "END":\n                    lines.append(line)\n                raw = "\\n".join(lines)\n                r = make_record(q, engine, raw, "blocked" if raw == "BLOCKED" else "ok")\n                urls = re.findall(r"https?://[^\\s<>\\]\\)]+", raw)\n                r["citations"] = [{"url": u.rstrip(".,;")} for u in sorted(set(urls))]\n                r["citation_coverage"] = "visible-only"\n                r["context"]["session"] = input("Collection session label (reuse for same account/session): ").strip() or "unknown"\n            with open(output, "a", encoding="utf-8") as stream:\n                stream.write(stable(r)+"\\n")\n            if r["status"] != "ok":\n                print("Stopped on unavailable/error response; no retries or paid fallback.")\n                return\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument("command", choices=["analyze", "collect", "battery"])\n    p.add_argument("--config", required=True)\n    p.add_argument("--captures")\n    p.add_argument("--out", default="output")\n    p.add_argument("--json", action="store_true")\n    p.add_argument("--engine", choices=sorted(ENGINES), default="chatgpt-consumer")\n    p.add_argument("--repeats", type=int, default=1)\n    p.add_argument("--model")\n    p.add_argument("--billing-disabled", action="store_true")\n    args = p.parse_args()\n    config = validate_config(json.loads(Path(args.config).read_text()))\n    if args.command == "battery":\n        print(json.dumps([{**q, "engine": e} for e in config["engines"] for q in config["queries"]], indent=2))\n        return\n    if not args.captures:\n        p.error("--captures is required")\n    if args.command == "collect":\n        collect(config, args.captures, args.engine, args.repeats, args.engine == "gemini-api", args.model, args.billing_disabled)\n    records, exclusions = read_records(args.captures, config)\n    analysis = analyze(config, records, exclusions)\n    if args.json:\n        print(stable(analysis))\n        return\n    out = Path(args.out)\n    out.mkdir(parents=True, exist_ok=True)\n    for name, content in [("analysis.json", json.dumps(analysis, indent=2, ensure_ascii=False)),\n                          ("opportunity-queue.json", json.dumps(analysis["opportunity_queue"], indent=2)),\n                          ("audit.md", report(analysis))]:\n        (out/name).write_text(content, encoding="utf-8")\n    print(f"Wrote audit and opportunity queue to {out}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
Path('target.json').write_text('{\n  "brand": "GoatCounter",\n  "category": "privacy-friendly website analytics",\n  "brands": [\n    {"name": "GoatCounter", "aliases": ["GoatCounter", "Goat Counter"], "domains": ["goatcounter.com"]},\n    {"name": "Plausible", "aliases": ["Plausible", "Plausible Analytics"], "domains": ["plausible.io"]},\n    {"name": "Fathom", "aliases": ["Fathom", "Fathom Analytics"], "domains": ["usefathom.com"]},\n    {"name": "Matomo", "aliases": ["Matomo"], "domains": ["matomo.org"]},\n    {"name": "Umami", "aliases": ["Umami"], "domains": ["umami.is"]},\n    {"name": "Simple Analytics", "aliases": ["Simple Analytics"], "domains": ["simpleanalytics.com"]},\n    {"name": "PostHog", "aliases": ["PostHog"], "domains": ["posthog.com"]},\n    {"name": "Usermaven", "aliases": ["Usermaven"], "domains": ["usermaven.com"]}\n  ],\n  "engines": ["chatgpt-consumer", "perplexity-consumer", "gemini-consumer"],\n  "queries": [\n    {"id": "saas-shortlist", "intent": 0.9, "fixability": 0.5, "fixability_reason": "Unvalidated prior: an accurate category comparison may be feasible; inclusion is not under brand control.", "framing": "business-shortlist", "prompt": "Which privacy-friendly website analytics tools should a small SaaS business shortlist as alternatives to Google Analytics? Recommend up to five brands, explain the tradeoffs briefly, and cite sources if available."},\n    {"id": "self-hosted-small", "intent": 0.8, "fixability": 0.5, "fixability_reason": "Unvalidated prior: document deployment constraints and supported use cases, subject to product verification.", "framing": "technical-constraints", "prompt": "I need simple privacy-friendly website analytics for a low-traffic site. I can self-host, prefer open source, and do not need individual user tracking. Which five tools should I compare before choosing? Explain tradeoffs briefly and cite sources if available."}\n  ],\n  "methodology": "Pilot convenience sample; repeated fresh chats in the same anonymous cloud-browser session. No independent-user, geographic or population-representativeness claim. Intent and fixability are declared operator priors, not measurements. Brand chosen after first category answer; selection bias disclosed.",\n  "brand_reference": "https://www.goatcounter.com/"\n}\n', encoding="utf-8")
Path('captures.jsonl').write_text('{"citation_coverage":"visible-only","citations":[{"url":"https://matomo.org/?utm_source=chatgpt.com"},{"url":"https://matomo.org/blog/2026/06/a-hands-on-guide-to-privacy-analytics/?utm_source=chatgpt.com"},{"url":"https://measr.app/compare/privacy-analytics/?utm_source=chatgpt.com"},{"url":"https://plausible.io/?utm_source=chatgpt.com"},{"url":"https://plausible.io/privacy-focused-web-analytics?utm_source=chatgpt.com"},{"url":"https://umami.is/?utm_source=chatgpt.com"},{"url":"https://usefathom.com/?utm_source=chatgpt.com"},{"url":"https://www.simpleanalytics.com/?utm_source=chatgpt.com"},{"url":"https://www.simpleanalytics.com/google-analytics-alternatives?utm_source=chatgpt.com"}],"context":{"collector":"browser-dom-snapshot","locale":"English","new_chat":true,"region":"unknown","search":"automatic; cited links visible","session":"anonymous-cloud-pilot-2026-09-07"},"engine":"chatgpt-consumer","evidence_kind":"measured","format":"dom-snapshot","id":"chatgpt-pilot-1","model":"ChatGPT (underlying model not disclosed)","observed_at":"2026-09-07T14:26:27.894Z","prompt":"Which privacy-friendly website analytics tools should a small SaaS business shortlist as alternatives to Google Analytics? Recommend up to five brands, explain the tradeoffs briefly, and cite sources if available.","provenance":{"clock":"browser collector clock; retained verbatim, not independently synchronized","note":"Anonymous chat URL is not a replay/share link. Snapshot is the preserved evidence.","snapshot":"capture-1.json","snapshot_sha256":"5e6f8aaacc7de205553e1265186236f1fce0f98cefe173fed1f64a49b429fc1f","surface_url":"https://chatgpt.com/"},"query_id":"saas-shortlist","raw_answer":"  - paragraph:\\n    - text: For a small SaaS business, I’d shortlist these\\n    - strong: five privacy-friendly Google Analytics alternatives\\n    - text: . The main tradeoff is usually\\n    - strong: privacy/simplicity vs. depth of user-level and product analytics\\n    - text: .\\n  - table:\\n    - rowgroup:\\n      - row \\"Tool Best fit Main tradeoff\\":\\n        - columnheader \\"Tool\\"\\n        - columnheader \\"Best fit\\"\\n        - columnheader \\"Main tradeoff\\"\\n    - rowgroup:\\n      - row \\"Plausible Analytics Best overall for a small SaaS marketing site Very simple and privacy-first, but deliberately lacks the user-level depth of GA4\\":\\n        - cell \\"Plausible Analytics\\":\\n          - strong:\\n            - generic: Plausible Analytics\\n        - cell \\"Best overall for a small SaaS marketing site\\"\\n        - cell \\"Very simple and privacy-first, but deliberately lacks the user-level depth of GA4\\"\\n      - row \\"Fathom Analytics Teams wanting a polished, low-maintenance hosted solution Excellent simplicity/privacy, but less extensible than heavier analytics platforms\\":\\n        - cell \\"Fathom Analytics\\":\\n          - strong:\\n            - generic: Fathom Analytics\\n        - cell \\"Teams wanting a polished, low-maintenance hosted solution\\"\\n        - cell \\"Excellent simplicity/privacy, but less extensible than heavier analytics platforms\\"\\n      - row \\"Matomo Businesses wanting GA-like depth and control Much more powerful, but consequently more complex to configure and operate\\":\\n        - cell \\"Matomo\\":\\n          - strong:\\n            - generic: Matomo\\n        - cell \\"Businesses wanting GA-like depth and control\\"\\n        - cell \\"Much more powerful, but consequently more complex to configure and operate\\"\\n      - row \\"Umami Developer-led SaaS teams that want self-hosting Lightweight and open source, but you take on infrastructure/maintenance\\":\\n        - cell \\"Umami\\":\\n          - strong:\\n            - generic: Umami\\n        - cell \\"Developer-led SaaS teams that want self-hosting\\"\\n        - cell \\"Lightweight and open source, but you take on infrastructure/maintenance\\"\\n      - row \\"Simple Analytics Teams that want extremely simple, aggregate website analytics Very privacy-friendly and easy, but intentionally limited for deep behavioral analysis\\":\\n        - cell \\"Simple Analytics\\":\\n          - strong:\\n            - generic: Simple Analytics\\n        - cell \\"Teams that want extremely simple, aggregate website analytics\\"\\n        - cell \\"Very privacy-friendly and easy, but intentionally limited for deep behavioral analysis\\"\\n  - button \\"Copy table\\":\\n  - heading \\"1. Plausible — my default recommendation\\" [level=3]\\n  - paragraph:\\n    - link \\"Plausible Analytics\\":\\n      - /url: https://plausible.io/?utm_source=chatgpt.com\\n      - text: Plausible Analytics\\n    - text: is probably the strongest starting point for most small SaaS companies. It is open source, uses no cookies or persistent identifiers, doesn\'t collect personal data, and hosts its cloud service on European infrastructure. Its dashboard is deliberately much simpler than GA4.\\n    - link \\"Plausible Analytics +1\\":\\n      - /url: https://plausible.io/privacy-focused-web-analytics?utm_source=chatgpt.com\\n      - generic: Plausible Analytics\\n      - generic: \\"+1\\"\\n  - paragraph:\\n    - strong: \\"Choose it if:\\"\\n    - text: you mainly need traffic, acquisition channels, pages, goals/events and conversions without building an analytics operation.\\n  - heading \\"2. Fathom — polished and hands-off\\" [level=3]\\n  - paragraph:\\n    - link \\"Fathom Analytics\\":\\n      - /url: https://usefathom.com/?utm_source=chatgpt.com\\n      - text: Fathom Analytics\\n    - text: \\"is another strong hosted option if you value a clean interface and minimal configuration. Its philosophy is similar to Plausible: privacy-first, cookieless analytics rather than detailed individual tracking. A recent comparison from Matomo places Fathom among the simpler cookieless privacy-focused platforms.\\"\\n    - link \\"Analytics Platform - Matomo\\":\\n      - /url: https://matomo.org/blog/2026/06/a-hands-on-guide-to-privacy-analytics/?utm_source=chatgpt.com\\n      - generic: Analytics Platform - Matomo\\n  - paragraph:\\n    - strong: \\"Tradeoff:\\"\\n    - text: you\'ll give up some of the customization and analytical depth available in Matomo or product-analytics platforms.\\n  - heading \\"3. Matomo — when you want more analytical power\\" [level=3]\\n  - paragraph:\\n    - link \\"Matomo\\":\\n      - /url: https://matomo.org/?utm_source=chatgpt.com\\n      - text: Matomo\\n    - text: is the option I\'d investigate if \\"alternative to GA4\\" means\\n    - strong: \\"\\\\\\"I want something nearly as capable, but with greater control over my data.\\\\\\"\\"\\n    - text: It supports substantially more functionality than lightweight privacy tools and can be self-hosted. Matomo describes its platform as privacy-focused and suitable from small organizations through enterprise.\\n    - link \\"Analytics Platform - Matomo\\":\\n      - /url: https://matomo.org/blog/2026/06/a-hands-on-guide-to-privacy-analytics/?utm_source=chatgpt.com\\n      - generic: Analytics Platform - Matomo\\n  - paragraph:\\n    - strong: \\"Tradeoff:\\"\\n    - text: more features mean more configuration and operational overhead. Self-hosting also means you\'re responsible for infrastructure and updates.\\n  - heading \\"4. Umami — excellent for technical/self-hosting teams\\" [level=3]\\n  - paragraph:\\n    - link \\"Umami\\":\\n      - /url: https://umami.is/?utm_source=chatgpt.com\\n      - text: Umami\\n    - text: is worth shortlisting if your SaaS has developers who are comfortable running infrastructure. It\'s an open-source, self-hostable approach that gives you considerable control over where analytics data lives. Current comparisons generally position it as a lightweight alternative rather than a full GA4 replacement.\\n    - link \\"Measr +1\\":\\n      - /url: https://measr.app/compare/privacy-analytics/?utm_source=chatgpt.com\\n      - generic: Measr\\n      - generic: \\"+1\\"\\n  - paragraph:\\n    - strong: \\"Tradeoff:\\"\\n    - text: the money you save on a hosted analytics subscription can become engineering/maintenance time.\\n  - heading \\"5. Simple Analytics — maximum simplicity\\" [level=3]\\n  - paragraph:\\n    - link \\"Simple Analytics\\":\\n      - /url: https://www.simpleanalytics.com/?utm_source=chatgpt.com\\n      - text: Simple Analytics\\n    - text: \\"takes perhaps the most minimalist approach: no cookies, no personal data, and a very straightforward dashboard. It\'s particularly attractive if you primarily care about aggregate website traffic rather than identifying and analyzing individual visitors.\\"\\n    - link \\"Simple Analytics +1\\":\\n      - /url: https://www.simpleanalytics.com/google-analytics-alternatives?utm_source=chatgpt.com\\n      - generic: Simple Analytics\\n      - generic: \\"+1\\"\\n  - paragraph:\\n    - strong: \\"Tradeoff:\\"\\n    - text: that simplicity means less granular behavioral analysis.\\n  - heading \\"What I\'d shortlist for a small SaaS\\" [level=3]\\n  - paragraph:\\n    - strong: \\"My order would be:\\"\\n  - list:\\n    - paragraph:\\n      - strong: Plausible\\n      - text: — best balance of privacy, simplicity and useful SaaS website metrics.\\n    - paragraph:\\n      - strong: Matomo\\n      - text: — if you need substantially more analytical depth.\\n    - paragraph:\\n      - strong: Fathom\\n      - text: — if you want the easiest polished hosted experience.\\n    - paragraph:\\n      - strong: Umami\\n      - text: — if self-hosting/data ownership is important.\\n    - paragraph:\\n      - strong: Simple Analytics\\n      - text: — if you want analytics to be almost completely \\"set and forget.\\"\\n  - paragraph:\\n    - text: \\"One important distinction:\\"\\n    - strong: website analytics isn\'t the same as product analytics.\\n    - text: If your real question is\\n    - emphasis: \\"\\\\\\"Which features do logged-in customers use, where do they drop out of onboarding, and which cohorts retain?\\\\\\"\\"\\n    - text: \\", I\'d also evaluate a product-analytics platform such as PostHog alongside one of the privacy-first website tools. Privacy-focused alternatives deliberately avoid much of the individual-level tracking that makes those analyses possible.\\"\\n    - link \\"Simple Analytics\\":\\n      - /url: https://www.simpleanalytics.com/google-analytics-alternatives?utm_source=chatgpt.com\\n      - generic: Simple Analytics\\n","sha256":"45895ada06fe54a5bfc283518570e773936d81835cb1566cb914a2fe2e30da53","status":"ok"}\n{"citation_coverage":"visible-only","citations":[{"url":"https://docs.umami.is/docs/about?utm_source=chatgpt.com"},{"url":"https://matomo.org/?utm_source=chatgpt.com"},{"url":"https://matomo.org/faq/general/faq_146/?utm_source=chatgpt.com"},{"url":"https://matomo.org/guide/manage-matomo/privacy/?utm_source=chatgpt.com"},{"url":"https://plausible.io/?utm_source=chatgpt.com"},{"url":"https://plausible.io/privacy-focused-web-analytics?utm_source=chatgpt.com"},{"url":"https://plausible.io/privacy?utm_source=chatgpt.com"},{"url":"https://umami.is/?utm_source=chatgpt.com"},{"url":"https://usefathom.com/?utm_source=chatgpt.com"},{"url":"https://www.simpleanalytics.com/?utm_source=chatgpt.com"},{"url":"https://www.simpleanalytics.com/security?utm_source=chatgpt.com"}],"context":{"collector":"browser-dom-snapshot","locale":"English","new_chat":true,"region":"unknown","search":"automatic; cited links visible","session":"anonymous-cloud-pilot-2026-09-07"},"engine":"chatgpt-consumer","evidence_kind":"measured","format":"dom-snapshot","id":"chatgpt-pilot-2","model":"ChatGPT (underlying model not disclosed)","observed_at":"2026-09-07T14:34:05.801Z","prompt":"Which privacy-friendly website analytics tools should a small SaaS business shortlist as alternatives to Google Analytics? Recommend up to five brands, explain the tradeoffs briefly, and cite sources if available.","provenance":{"clock":"browser collector clock; retained verbatim, not independently synchronized","note":"Anonymous chat URL is not a replay/share link. Snapshot is the preserved evidence.","snapshot":"capture-2.json","snapshot_sha256":"752bfce647910b6971a2c74e00e77b17d183431b831566d26e77e75aa93d7a5a","surface_url":"https://chatgpt.com/"},"query_id":"saas-shortlist","raw_answer":"  - paragraph:\\n    - text: For a small SaaS, I’d shortlist these five. The key tradeoff is\\n    - strong: privacy/simplicity vs. depth of user-level product analytics\\n    - text: —none is a perfect 1:1 replacement for everything GA4 can do.\\n  - table:\\n    - rowgroup:\\n      - row \\"Tool Best for Main tradeoff\\":\\n        - columnheader \\"Tool\\"\\n        - columnheader \\"Best for\\"\\n        - columnheader \\"Main tradeoff\\"\\n    - rowgroup:\\n      - row \\"Plausible Best overall GA replacement Very simple and privacy-friendly, but intentionally lacks GA4-style user-level analysis\\":\\n        - cell \\"Plausible\\":\\n          - strong:\\n            - link \\"Plausible\\":\\n              - /url: https://plausible.io/?utm_source=chatgpt.com\\n              - text: Plausible\\n        - cell \\"Best overall GA replacement\\"\\n        - cell \\"Very simple and privacy-friendly, but intentionally lacks GA4-style user-level analysis\\"\\n      - row \\"Fathom Analytics Polished, low-maintenance analytics Excellent simplicity/privacy, but less customizable than heavier analytics platforms\\":\\n        - cell \\"Fathom Analytics\\":\\n          - strong:\\n            - link \\"Fathom Analytics\\":\\n              - /url: https://usefathom.com/?utm_source=chatgpt.com\\n              - text: Fathom Analytics\\n        - cell \\"Polished, low-maintenance analytics\\"\\n        - cell \\"Excellent simplicity/privacy, but less customizable than heavier analytics platforms\\"\\n      - row \\"Umami Technical teams wanting control Open source and self-hostable, but self-hosting means you own the infrastructure/maintenance\\":\\n        - cell \\"Umami\\":\\n          - strong:\\n            - link \\"Umami\\":\\n              - /url: https://umami.is/?utm_source=chatgpt.com\\n              - text: Umami\\n        - cell \\"Technical teams wanting control\\"\\n        - cell \\"Open source and self-hostable, but self-hosting means you own the infrastructure/maintenance\\"\\n      - row \\"Matomo Closest privacy-conscious alternative to GA4 Much deeper feature set, but consequently more complex; some configurations use cookies/require consent\\":\\n        - cell \\"Matomo\\":\\n          - strong:\\n            - link \\"Matomo\\":\\n              - /url: https://matomo.org/?utm_source=chatgpt.com\\n              - text: Matomo\\n        - cell \\"Closest privacy-conscious alternative to GA4\\"\\n        - cell \\"Much deeper feature set, but consequently more complex; some configurations use cookies/require consent\\"\\n      - row \\"Simple Analytics Maximum simplicity/privacy Aggregate traffic analytics rather than detailed individual-user journeys\\":\\n        - cell \\"Simple Analytics\\":\\n          - strong:\\n            - link \\"Simple Analytics\\":\\n              - /url: https://www.simpleanalytics.com/?utm_source=chatgpt.com\\n              - text: Simple Analytics\\n        - cell \\"Maximum simplicity/privacy\\"\\n        - cell \\"Aggregate traffic analytics rather than detailed individual-user journeys\\"\\n  - button \\"Copy table\\":\\n  - heading \\"My take\\" [level=3]\\n  - paragraph:\\n    - strong: 1. Plausible — best default for most small SaaS businesses.\\n    - text: Plausible is open source, EU-based, cookieless, and says it collects no personal data and doesn\'t sell visitor information. Its interface and implementation are deliberately lightweight.\\n    - link \\"Plausible Analytics +2 Plausible Analytics +2\\":\\n      - /url: https://plausible.io/privacy?utm_source=chatgpt.com\\n      - generic: Plausible Analytics\\n      - generic: \\"+2\\"\\n      - generic: Plausible Analytics\\n      - generic: \\"+2\\"\\n    - strong: \\"Choose it if:\\"\\n    - text: you primarily need traffic, acquisition channels, pages, goals/events, and a clean dashboard.\\n  - paragraph:\\n    - strong: 2. Fathom — best polished hosted option.\\n    - text: I\'d put Fathom alongside Plausible if your priority is\\n    - emphasis: \\"\\\\\\"install it and stop thinking about analytics infrastructure.\\\\\\"\\"\\n    - text: The tradeoff is that privacy-first simplicity means fewer of the elaborate segmentation and user-journey capabilities associated with GA4.\\n  - paragraph:\\n    - strong: 3. Umami — best for developers/self-hosting.\\n    - text: Umami is open source and can be self-hosted, giving you control over where the analytics data lives. It supports custom events, funnels, retention, UTM tracking, goals and APIs while remaining cookieless/privacy-focused.\\n    - link \\"Umami Docs\\":\\n      - /url: https://docs.umami.is/docs/about?utm_source=chatgpt.com\\n      - generic: Umami Docs\\n    - strong: \\"Choose it if:\\"\\n    - text: your SaaS has engineering capacity and you value data ownership more than having a completely managed service.\\n  - paragraph:\\n    - strong: 4. Matomo — best when you need GA-like depth.\\n    - text: \\"Matomo is the heavyweight choice here: it offers much more configurability and control than Plausible/Fathom. Its privacy tooling lets you configure anonymization, cookies, consent and data retention.\\"\\n    - link \\"Analytics Platform - Matomo\\":\\n      - /url: https://matomo.org/guide/manage-matomo/privacy/?utm_source=chatgpt.com\\n      - generic: Analytics Platform - Matomo\\n    - text: \\"The catch is important:\\"\\n    - strong: Matomo isn\'t automatically cookieless\\n    - text: —its standard JavaScript tracking uses first-party cookies, although it can be configured differently.\\n    - link \\"Analytics Platform - Matomo\\":\\n      - /url: https://matomo.org/faq/general/faq_146/?utm_source=chatgpt.com\\n      - generic: Analytics Platform - Matomo\\n    - strong: \\"Choose it if:\\"\\n    - text: you need sophisticated analytics and are willing to accept more configuration/maintenance.\\n  - paragraph:\\n    - strong: 5. Simple Analytics — best if you want \\"just the useful numbers.\\"\\n    - text: It deliberately avoids cookies, visitor IDs, personal data and cross-session tracking, with EU hosting.\\n    - link \\"Simple Analytics +1\\":\\n      - /url: https://www.simpleanalytics.com/security?utm_source=chatgpt.com\\n      - generic: Simple Analytics\\n      - generic: \\"+1\\"\\n    - text: That makes it particularly attractive if privacy/compliance simplicity is more important than behavioral depth.\\n    - strong: \\"Choose it if:\\"\\n    - text: you mainly care about aggregate traffic, referrals, campaigns and conversions rather than reconstructing individual customer journeys.\\n  - heading \\"What I\'d shortlist for a SaaS specifically\\" [level=3]\\n  - paragraph:\\n    - strong: Plausible + Umami + Matomo\\n    - text: \\"would be my first three to actually trial:\\"\\n  - list:\\n    - paragraph:\\n      - strong: Plausible\\n      - text: → easiest all-around replacement.\\n    - paragraph:\\n      - strong: Umami\\n      - text: → strongest option if you want to own/self-host the data.\\n    - paragraph:\\n      - strong: Matomo\\n      - text: → strongest if you need significantly more analytical depth.\\n  - paragraph:\\n    - text: \\"One important caveat:\\"\\n    - strong: website analytics ≠ product analytics\\n    - text: . If your real requirement is\\n    - emphasis: \\"\\\\\\"Which features do logged-in customers use? Where do they drop out of onboarding? What are retention cohorts?\\\\\\"\\"\\n    - text: \\", I\'d also evaluate a privacy-conscious product-analytics platform rather than assuming a GA replacement is sufficient.\\"\\n  - paragraph:\\n    - text: For a small SaaS whose priority is\\n    - strong: privacy + low maintenance + useful marketing analytics\\n    - text: \\", though,\\"\\n    - strong: Plausible would be my default pick\\n    - text: .\\n    - link \\"Plausible Analytics\\":\\n      - /url: https://plausible.io/privacy-focused-web-analytics?utm_source=chatgpt.com\\n      - generic: Plausible Analytics\\n","sha256":"c42c8cdc511341ca16d38ad67bdbcf9b20ea6f428287697288cb99adea289fae","status":"ok"}\n{"citation_coverage":"visible-only","citations":[{"url":"https://docs.umami.is/docs/about?utm_source=chatgpt.com"},{"url":"https://helionlabs.dev/articles/self-hosted-web-analytics?utm_source=chatgpt.com"},{"url":"https://plausible.io/about?utm_source=chatgpt.com"},{"url":"https://toolradar.com/guides/best-self-hosted-analytics?utm_source=chatgpt.com"},{"url":"https://www.goatcounter.com/?utm_source=chatgpt.com"},{"url":"https://www.ssdnodes.com/learn/self-hosted-web-analytics-vps?utm_source=chatgpt.com"}],"context":{"collector":"browser-dom-snapshot","locale":"English","new_chat":true,"region":"unknown","search":"automatic; cited links visible","session":"anonymous-cloud-pilot-2026-09-07"},"engine":"chatgpt-consumer","evidence_kind":"measured","format":"dom-snapshot","id":"chatgpt-pilot-3","model":"ChatGPT (underlying model not disclosed)","observed_at":"2026-09-07T14:36:47.386Z","prompt":"I need simple privacy-friendly website analytics for a low-traffic site. I can self-host, prefer open source, and do not need individual user tracking. Which five tools should I compare before choosing? Explain tradeoffs briefly and cite sources if available.","provenance":{"clock":"browser collector clock; retained verbatim, not independently synchronized","note":"Anonymous chat URL is not a replay/share link. Snapshot is the preserved evidence.","snapshot":"capture-3.json","snapshot_sha256":"63862365695f7e73faa105a1ac6444627e37c3f6c9fb2dd086d1d732731cba8a","surface_url":"https://chatgpt.com/"},"query_id":"self-hosted-small","raw_answer":"  - paragraph:\\n    - text: For your requirements—\\n    - strong: low traffic, self-hosted, open source, privacy-first, and no individual-user tracking\\n    - text: \\"—I’d compare these five:\\"\\n  - table:\\n    - rowgroup:\\n      - row \\"Tool Why consider it Main tradeoff\\":\\n        - columnheader \\"Tool\\"\\n        - columnheader \\"Why consider it\\"\\n        - columnheader \\"Main tradeoff\\"\\n    - rowgroup:\\n      - row \\"GoatCounter Probably the best fit for a small site. Very lightweight, privacy-oriented, doesn\'t use unique user identifiers, and can even work via a tracking pixel or server/log integration. GoatCounter +1 Intentionally minimalist; fewer sophisticated reports and integrations than the bigger platforms.\\":\\n        - cell \\"GoatCounter\\":\\n          - strong:\\n            - generic: GoatCounter\\n        - cell \\"Probably the best fit for a small site. Very lightweight, privacy-oriented, doesn\'t use unique user identifiers, and can even work via a tracking pixel or server/log integration. GoatCounter +1\\":\\n          - text: Probably the best fit for a small site. Very lightweight, privacy-oriented, doesn\'t use unique user identifiers, and can even work via a tracking pixel or server/log integration.\\n          - link \\"GoatCounter +1\\":\\n            - /url: https://www.goatcounter.com/?utm_source=chatgpt.com\\n            - generic: GoatCounter\\n            - generic: \\"+1\\"\\n        - cell \\"Intentionally minimalist; fewer sophisticated reports and integrations than the bigger platforms.\\"\\n      - row \\"Umami Excellent balance of simplicity and features. It is cookie-free, doesn\'t collect personal data, is self-hostable, and supports custom events, goals and UTM tracking. Umami Docs +1 More infrastructure than GoatCounter—typically Docker plus a database—and its broader feature set may be unnecessary for a simple brochure/content site.\\":\\n        - cell \\"Umami\\":\\n          - strong:\\n            - generic: Umami\\n        - cell \\"Excellent balance of simplicity and features. It is cookie-free, doesn\'t collect personal data, is self-hostable, and supports custom events, goals and UTM tracking. Umami Docs +1\\":\\n          - text: Excellent balance of simplicity and features. It is cookie-free, doesn\'t collect personal data, is self-hostable, and supports custom events, goals and UTM tracking.\\n          - link \\"Umami Docs +1\\":\\n            - /url: https://docs.umami.is/docs/about?utm_source=chatgpt.com\\n            - generic: Umami Docs\\n            - generic: \\"+1\\"\\n        - cell \\"More infrastructure than GoatCounter—typically Docker plus a database—and its broader feature set may be unnecessary for a simple brochure/content site.\\"\\n      - \'row \\"Plausible Analytics Very polished, simple dashboard and privacy model: no cookies, personal data, cross-site tracking or user profiles. Its Community Edition is AGPLv3 and self-hostable. Plausible Analytics +1 Self-hosting is somewhat heavier than the minimalist options; AGPL is also worth considering if licensing matters to you.\\"\':\\n        - cell \\"Plausible Analytics\\":\\n          - strong:\\n            - generic: Plausible Analytics\\n        - \'cell \\"Very polished, simple dashboard and privacy model: no cookies, personal data, cross-site tracking or user profiles. Its Community Edition is AGPLv3 and self-hostable. Plausible Analytics +1\\"\':\\n          - text: \\"Very polished, simple dashboard and privacy model: no cookies, personal data, cross-site tracking or user profiles. Its Community Edition is AGPLv3 and self-hostable.\\"\\n          - link \\"Plausible Analytics +1\\":\\n            - /url: https://plausible.io/about?utm_source=chatgpt.com\\n            - generic: Plausible Analytics\\n            - generic: \\"+1\\"\\n        - cell \\"Self-hosting is somewhat heavier than the minimalist options; AGPL is also worth considering if licensing matters to you.\\"\\n      - row \\"Ackee A deliberately simple, privacy-focused self-hosted analytics option. MIT-licensed and aimed at straightforward website statistics. Helion Labs Smaller ecosystem and less feature depth; its MongoDB dependency adds another component to maintain. Toolradar\\":\\n        - cell \\"Ackee\\":\\n          - strong:\\n            - generic: Ackee\\n        - cell \\"A deliberately simple, privacy-focused self-hosted analytics option. MIT-licensed and aimed at straightforward website statistics. Helion Labs\\":\\n          - text: A deliberately simple, privacy-focused self-hosted analytics option. MIT-licensed and aimed at straightforward website statistics.\\n          - link \\"Helion Labs\\":\\n            - /url: https://helionlabs.dev/articles/self-hosted-web-analytics?utm_source=chatgpt.com\\n            - generic: Helion Labs\\n        - cell \\"Smaller ecosystem and less feature depth; its MongoDB dependency adds another component to maintain. Toolradar\\":\\n          - text: Smaller ecosystem and less feature depth; its MongoDB dependency adds another component to maintain.\\n          - link \\"Toolradar\\":\\n            - /url: https://toolradar.com/guides/best-self-hosted-analytics?utm_source=chatgpt.com\\n            - generic: Toolradar\\n      - \'row \\"Matomo The choice if you might eventually want a full Google Analytics replacement: much deeper reporting and functionality, while still being self-hostable and GPL-licensed. Helion Labs +1 Probably overkill for your stated use case, with substantially more operational complexity than GoatCounter, Umami or Plausible.\\"\':\\n        - cell \\"Matomo\\":\\n          - strong:\\n            - generic: Matomo\\n        - \'cell \\"The choice if you might eventually want a full Google Analytics replacement: much deeper reporting and functionality, while still being self-hostable and GPL-licensed. Helion Labs +1\\"\':\\n          - text: The choice if you might eventually want a\\n          - strong: full Google Analytics replacement\\n          - text: \\": much deeper reporting and functionality, while still being self-hostable and GPL-licensed.\\"\\n          - link \\"Helion Labs +1\\":\\n            - /url: https://helionlabs.dev/articles/self-hosted-web-analytics?utm_source=chatgpt.com\\n            - generic: Helion Labs\\n            - generic: \\"+1\\"\\n        - cell \\"Probably overkill for your stated use case, with substantially more operational complexity than GoatCounter, Umami or Plausible.\\"\\n  - button \\"Copy table\\":\\n  - heading \\"My shortlist for your case\\" [level=3]:\\n    - text: My shortlist for\\n    - emphasis: your\\n    - text: case\\n  - paragraph: \\"I\'d test them in this order:\\"\\n  - list:\\n    - paragraph:\\n      - strong: GoatCounter\\n      - text: — if you genuinely only want basic traffic/page/referrer statistics.\\n    - paragraph:\\n      - strong: Umami\\n      - text: — if you want a nicer dashboard and occasional custom events.\\n    - paragraph:\\n      - strong: Plausible CE\\n      - text: — if polish and a very opinionated privacy model matter most.\\n    - paragraph:\\n      - strong: Ackee\\n      - text: — worth trying if you prefer its minimal approach/stack.\\n    - paragraph:\\n      - strong: Matomo\\n      - text: — mainly as a benchmark for what you\\n      - emphasis: could\\n      - text: have, rather than what you probably need.\\n  - paragraph:\\n    - text: \\"One useful distinction:\\"\\n    - strong: privacy-friendly doesn\'t necessarily mean “no analytics about visits.”\\n    - text: These tools can still aggregate things such as pages, referrers, devices or countries while avoiding persistent individual identifiers. GoatCounter, for example, explicitly says it doesn\'t track users with unique identifiers, while Umami says it avoids cookies, fingerprinting and personal-data collection.\\n    - link \\"GoatCounter +1\\":\\n      - /url: https://www.goatcounter.com/?utm_source=chatgpt.com\\n      - generic: GoatCounter\\n      - generic: \\"+1\\"\\n  - paragraph:\\n    - text: For a genuinely low-traffic site, I\'d favor\\n    - strong: the smallest operational footprint over maximum features\\n    - text: . Recent comparisons similarly put GoatCounter among the easiest/lightest choices, with Umami offering a good middle ground and Matomo at the feature-heavy end.\\n    - link \\"SSD Nodes +1\\":\\n      - /url: https://www.ssdnodes.com/learn/self-hosted-web-analytics-vps?utm_source=chatgpt.com\\n      - generic: SSD Nodes\\n      - generic: \\"+1\\"\\n  - paragraph:\\n    - text: If you tell me your hosting environment (e.g.\\n    - strong: Docker/VPS, shared hosting, or static site + serverless\\n    - text: ), I can narrow these five down to the best\\n    - strong: 2–3\\n    - text: for you.\\n","sentiment_reviews":{"GoatCounter":{"label":"positive","note":"Positive recommendation in this answer; not a verified product-quality claim.","quote":"Probably the best fit for a small site.","reviewer":"Codex analyst"}},"sha256":"8b3f855a82537a50f790b039fd57583965d1340be42b362979c946ea65da4392","status":"ok"}\n{"citation_coverage":"visible-only","citations":[{"url":"https://docs.umami.is/docs?utm_source=chatgpt.com"},{"url":"https://github.com/electerious/ackee?utm_source=chatgpt.com"},{"url":"https://matomo.org/privacy/?utm_source=chatgpt.com"},{"url":"https://plausible.github.io/docs/self-hosting?utm_source=chatgpt.com"},{"url":"https://plausible.io/about?utm_source=chatgpt.com"},{"url":"https://www.goatcounter.com/?utm_source=chatgpt.com"},{"url":"https://www.goatcounter.com/help/privacy?utm_source=chatgpt.com"}],"context":{"collector":"browser-dom-snapshot","locale":"English","new_chat":true,"region":"unknown","search":"automatic; cited links visible","session":"anonymous-cloud-pilot-2026-09-07"},"engine":"chatgpt-consumer","evidence_kind":"measured","format":"dom-snapshot","id":"chatgpt-pilot-4","model":"ChatGPT (underlying model not disclosed)","observed_at":"2026-09-07T14:39:09.528Z","prompt":"I need simple privacy-friendly website analytics for a low-traffic site. I can self-host, prefer open source, and do not need individual user tracking. Which five tools should I compare before choosing? Explain tradeoffs briefly and cite sources if available.","provenance":{"clock":"browser collector clock; retained verbatim, not independently synchronized","note":"Anonymous chat URL is not a replay/share link. Snapshot is the preserved evidence.","snapshot":"capture-4.json","snapshot_sha256":"10c6b93ce9e6564116272039007223381d793eb930fef34dcb8ba373717d9b9b","surface_url":"https://chatgpt.com/"},"query_id":"self-hosted-small","raw_answer":"  - paragraph:\\n    - text: For your requirements—\\n    - strong: low traffic, self-hosting, open source, privacy-friendly, and no individual-user tracking\\n    - text: \\"—I’d compare these five:\\"\\n  - table:\\n    - rowgroup:\\n      - row \\"Tool Why consider it Main tradeoff\\":\\n        - columnheader \\"Tool\\"\\n        - columnheader \\"Why consider it\\"\\n        - columnheader \\"Main tradeoff\\"\\n    - rowgroup:\\n      - row \\"GoatCounter Probably the closest fit. Very lightweight, deliberately avoids unique user identifiers, and can be self-hosted. It can even work via a tracking pixel or server-side/log-based collection. GoatCounter +1 Much less feature-rich than the bigger analytics platforms.\\":\\n        - cell \\"GoatCounter\\":\\n          - strong:\\n            - generic: GoatCounter\\n        - cell \\"Probably the closest fit. Very lightweight, deliberately avoids unique user identifiers, and can be self-hosted. It can even work via a tracking pixel or server-side/log-based collection. GoatCounter +1\\":\\n          - text: Probably the closest fit. Very lightweight, deliberately avoids unique user identifiers, and can be self-hosted. It can even work via a tracking pixel or server-side/log-based collection.\\n          - link \\"GoatCounter +1\\":\\n            - /url: https://www.goatcounter.com/?utm_source=chatgpt.com\\n            - generic: GoatCounter\\n            - generic: \\"+1\\"\\n        - cell \\"Much less feature-rich than the bigger analytics platforms.\\"\\n      - row \\"Umami Excellent balance of simplicity and capability. No cookies/personal data by default, self-hostable, lightweight, and includes useful things like referrers, devices, events and goals. Umami Docs +1 More infrastructure than GoatCounter, and its feature set can be more than you need.\\":\\n        - cell \\"Umami\\":\\n          - strong:\\n            - generic: Umami\\n        - cell \\"Excellent balance of simplicity and capability. No cookies/personal data by default, self-hostable, lightweight, and includes useful things like referrers, devices, events and goals. Umami Docs +1\\":\\n          - text: Excellent balance of simplicity and capability. No cookies/personal data by default, self-hostable, lightweight, and includes useful things like referrers, devices, events and goals.\\n          - link \\"Umami Docs +1\\":\\n            - /url: https://docs.umami.is/docs?utm_source=chatgpt.com\\n            - generic: Umami Docs\\n            - generic: \\"+1\\"\\n        - cell \\"More infrastructure than GoatCounter, and its feature set can be more than you need.\\"\\n      - \'row \\"Plausible Analytics Very polished and simple. No cookies, personal data, cross-site tracking or user profiles; its Community Edition is self-hostable and AGPLv3. Plausible Analytics +1 Self-hosting is relatively heavier: the official setup uses Docker plus PostgreSQL and ClickHouse. Plausible\\"\':\\n        - cell \\"Plausible Analytics\\":\\n          - strong:\\n            - generic: Plausible Analytics\\n        - cell \\"Very polished and simple. No cookies, personal data, cross-site tracking or user profiles; its Community Edition is self-hostable and AGPLv3. Plausible Analytics +1\\":\\n          - text: Very polished and simple. No cookies, personal data, cross-site tracking or user profiles; its Community Edition is self-hostable and AGPLv3.\\n          - link \\"Plausible Analytics +1\\":\\n            - /url: https://plausible.io/about?utm_source=chatgpt.com\\n            - generic: Plausible Analytics\\n            - generic: \\"+1\\"\\n        - \'cell \\"Self-hosting is relatively heavier: the official setup uses Docker plus PostgreSQL and ClickHouse. Plausible\\"\':\\n          - text: \\"Self-hosting is relatively heavier: the official setup uses Docker plus PostgreSQL and ClickHouse.\\"\\n          - link \\"Plausible\\":\\n            - /url: https://plausible.github.io/docs/self-hosting?utm_source=chatgpt.com\\n            - generic: Plausible\\n      - row \\"Matomo The most capable option here if you might eventually want detailed analytics. It\'s open source, self-hostable, and has extensive privacy/anonymization controls. Analytics Platform - Matomo +1 Likely overkill for a low-traffic site; it is designed much more like a full Google Analytics replacement.\\":\\n        - cell \\"Matomo\\":\\n          - strong:\\n            - generic: Matomo\\n        - cell \\"The most capable option here if you might eventually want detailed analytics. It\'s open source, self-hostable, and has extensive privacy/anonymization controls. Analytics Platform - Matomo +1\\":\\n          - text: The most capable option here if you might eventually want detailed analytics. It\'s open source, self-hostable, and has extensive privacy/anonymization controls.\\n          - link \\"Analytics Platform - Matomo +1\\":\\n            - /url: https://matomo.org/privacy/?utm_source=chatgpt.com\\n            - generic: Analytics Platform - Matomo\\n            - generic: \\"+1\\"\\n        - cell \\"Likely overkill for a low-traffic site; it is designed much more like a full Google Analytics replacement.\\":\\n          - text: Likely\\n          - strong: overkill for a low-traffic site\\n          - text: ; it is designed much more like a full Google Analytics replacement.\\n      - row \\"Ackee Minimal, open source and self-hosted; no cookies or unique-user tracking, with a clean interface and GraphQL API. GitHub +1 Requires Node.js + MongoDB and intentionally has fewer analytics capabilities than Matomo/Umami. GitHub\\":\\n        - cell \\"Ackee\\":\\n          - strong:\\n            - generic: Ackee\\n        - cell \\"Minimal, open source and self-hosted; no cookies or unique-user tracking, with a clean interface and GraphQL API. GitHub +1\\":\\n          - text: Minimal, open source and self-hosted; no cookies or unique-user tracking, with a clean interface and GraphQL API.\\n          - link \\"GitHub +1\\":\\n            - /url: https://github.com/electerious/ackee?utm_source=chatgpt.com\\n            - generic: GitHub\\n            - generic: \\"+1\\"\\n        - cell \\"Requires Node.js + MongoDB and intentionally has fewer analytics capabilities than Matomo/Umami. GitHub\\":\\n          - text: Requires Node.js + MongoDB and intentionally has fewer analytics capabilities than Matomo/Umami.\\n          - link \\"GitHub\\":\\n            - /url: https://github.com/electerious/ackee?utm_source=chatgpt.com\\n            - generic: GitHub\\n  - button \\"Copy table\\":\\n  - heading \\"My shortlist for your particular use case\\" [level=3]\\n  - paragraph: \\"I\'d test them in this order:\\"\\n  - list:\\n    - paragraph:\\n      - strong: GoatCounter\\n      - text: — best if you genuinely only want basic traffic/pageview statistics.\\n    - paragraph:\\n      - strong: Umami\\n      - text: — best overall balance if you want a nicer dashboard and occasional events/goals.\\n    - paragraph:\\n      - strong: Plausible\\n      - text: — excellent UX, but I\'d question the additional infrastructure for a tiny site.\\n    - paragraph:\\n      - strong: Ackee\\n      - text: — worth considering if you like its minimalist approach and Node/Mongo stack.\\n    - paragraph:\\n      - strong: Matomo\\n      - text: — choose this primarily if you anticipate needing much deeper analytics later.\\n  - paragraph:\\n    - strong: \\"One important distinction:\\"\\n    - text: “privacy-friendly” doesn\'t necessarily mean “doesn\'t track users.” Matomo, for example, is capable of extensive user-activity analytics but gives you controls to anonymize/configure what it collects.\\n    - link \\"Analytics Platform - Matomo\\":\\n      - /url: https://matomo.org/privacy/?utm_source=chatgpt.com\\n      - generic: Analytics Platform - Matomo\\n    - text: \\"GoatCounter, by contrast, is much closer to your stated philosophy: its documentation says it doesn\'t store IP addresses, full User-Agent strings, cookies, or tracker IDs in its normal privacy-oriented setup.\\"\\n    - link \\"GoatCounter\\":\\n      - /url: https://www.goatcounter.com/help/privacy?utm_source=chatgpt.com\\n      - generic: GoatCounter\\n  - paragraph:\\n    - text: For a\\n    - strong: small personal/project site where you mainly want “How many people visited, which pages, and where did they come from?”\\n    - text: \\", I\'d start with\\"\\n    - strong: GoatCounter vs. Umami\\n    - text: .\\n","sha256":"39dd4a8f51ffba1a4327f974c1b8c2040e5f19976ab4e238e9746180aa495076","status":"ok"}\n{"citation_coverage":"visible-only","citations":[{"url":"https://usermaven.com/blog/privacy-first-analytics-tools#:~:text=Event%20tracking.%20Anonymized%20data%20collection.%20Referrer%20and,and%20pageview%20reporting.%20Goal%20and%20event%20tracking."}],"context":{"collector":"browser-dom-snapshot","locale":"English","new_chat":true,"region":"unknown","search":"automatic; source details visible","session":"anonymous-cloud-pilot-2026-09-07"},"engine":"gemini-consumer","evidence_kind":"measured","format":"dom-snapshot","id":"gemini-pilot-1","model":"Flash-Lite (UI label; version not disclosed)","observed_at":"2026-09-07T14:40:06.455Z","prompt":"Which privacy-friendly website analytics tools should a small SaaS business shortlist as alternatives to Google Analytics? Recommend up to five brands, explain the tradeoffs briefly, and cite sources if available.","provenance":{"clock":"browser collector clock; retained verbatim, not independently synchronized","note":"Only one expanded citation drawer captured; source coverage is partial.","snapshot":"gemini-1.json","snapshot_sha256":"3266992058e0516f6976969472bb97896c1fa83ec9416ed5ec593c0387de8453","surface_url":"https://gemini.google.com/app/abb4a03f55eaeb68"},"query_id":"saas-shortlist","raw_answer":"  - paragraph: \\"For a small SaaS business, moving away from Google Analytics usually means balancing two competing needs: strict privacy compliance (avoiding annoying cookie banners and respecting user data) and product/marketing depth (needing conversion funnels, event tracking, and attribution rather than just basic pageviews).\\"\\n  - paragraph: Here are five privacy-friendly alternatives to shortlist, categorized by their strengths and brief tradeoffs.\\n  - heading \\"1. Plausible Analytics\\" [level=3]\\n  - list:\\n    - paragraph:\\n      - text: \\"Best for:\\"\\n      - text: A clean, lightweight dashboard focused on core website traffic and marketing metrics.\\n      - superscript\\n      - button \\"View source details for citation from Usermaven. Press Enter to open sources dialog.\\" [expanded] [active]:\\n        - generic: Usermaven\\n      - dialog:\\n        - \'link \\"Usermaven 10 privacy-first analytics tools worth exploring in 2026 - Usermaven “Event tracking. Anonymized data collection. Referrer and campaign data reporting. Open-source with a self-hosted option. Best for: Developers and small teams th...”\\"\':\\n          - /url: https://usermaven.com/blog/privacy-first-analytics-tools#:~:text=Event%20tracking.%20Anonymized%20data%20collection.%20Referrer%20and,and%20pageview%20reporting.%20Goal%20and%20event%20tracking.\\n          - generic: Usermaven\\n          - generic: 10 privacy-first analytics tools worth exploring in 2026 - Usermaven\\n          - generic: \\"“Event tracking. Anonymized data collection. Referrer and campaign data reporting. Open-source with a self-hosted option. Best for: Developers and small teams th...”\\"\\n    - paragraph: \\"The Tradeoff: Plausible is intentionally minimalist. It handles custom events and goal tracking well, but it does not provide deep user-level product analytics, session replays, or complex multi-touch attribution out of the box.\\"\\n    - paragraph:\\n      - text: \\"Privacy Model:\\"\\n      - text: Fully cookieless, open-source, hosted in the EU, and compliant with GDPR, CCPA, and PECR without needing a cookie banner.\\n      - superscript\\n      - button \\"View source details for citation from bootstrap.build. Press Enter to open sources dialog.\\":\\n        - generic: bootstrap.build\\n  - heading \\"2. PostHog\\" [level=3]\\n  - list:\\n    - paragraph:\\n      - text: \\"Best for:\\"\\n      - text: SaaS businesses that need all-in-one product analytics (funnels, retention, feature flags, and session replays).\\n      - superscript\\n      - button \\"View source details for citation from bootstrap.build. Press Enter to open sources dialog.\\":\\n        - generic: bootstrap.build\\n    - paragraph:\\n      - text: \\"The Tradeoff: Because it captures deep behavioral and user-level data, it has a steeper learning curve than simple traffic counters.\\"\\n      - text: Furthermore, if you use their cloud offering extensively, costs scale with usage (though they feature a very generous free tier).\\n      - superscript\\n      - button \\"View source details for citation from bootstrap.build. Press Enter to open sources dialog.\\":\\n        - generic: bootstrap.build\\n    - paragraph:\\n      - text: \\"Privacy Model: Fully customizable privacy controls;\\"\\n      - text: because it is open-source, you can self-host it entirely on your own infrastructure to keep 100% data ownership.\\n      - superscript\\n      - button \\"View source details for citation from bootstrap.build. Press Enter to open sources dialog.\\":\\n        - generic: bootstrap.build\\n  - heading \\"3. Simple Analytics\\" [level=3]\\n  - list:\\n    - paragraph:\\n      - text: \\"Best for:\\"\\n      - text: Clean, straightforward marketing reports with a zero-config setup.\\n      - superscript\\n      - button \\"View source details for citation from bootstrap.build. Press Enter to open sources dialog.\\":\\n        - generic: bootstrap.build\\n    - paragraph:\\n      - text: \\"The Tradeoff:\\"\\n      - text: Like Plausible, it\'s designed primarily for web traffic and lightweight event tracking.\\n      - superscript\\n      - text: It lacks advanced SaaS product metrics like deep cohort retention grids or user-journey pathing.\\n      - button \\"View source details for citation from Usermaven. Press Enter to open sources dialog.\\":\\n        - generic: Usermaven\\n    - paragraph:\\n      - text: \\"Privacy Model:\\"\\n      - text: 100% GDPR-compliant, cookieless, and stores data securely in the Netherlands (EU).\\n      - superscript\\n      - text: They heavily prioritize bypassing ad-blockers ethically while keeping data anonymized.\\n      - button \\"View source details for citation from Simple Analytics. Press Enter to open sources dialog.\\":\\n        - generic: Simple Analytics\\n  - heading \\"4. Usermaven\\" [level=3]\\n  - list:\\n    - paragraph:\\n      - text: \\"Best for:\\"\\n      - text: Growth-focused SaaS teams that want to bridge website traffic with marketing attribution and product event tracking.\\n      - superscript\\n      - button \\"View source details for citation from Usermaven. Press Enter to open sources dialog.\\":\\n        - generic: Usermaven\\n    - paragraph: \\"The Tradeoff: It is a newer player compared to legacy tools like Matomo, meaning its community documentation and ecosystem integrations are still maturing.\\"\\n    - paragraph:\\n      - text: \\"Privacy Model:\\"\\n      - text: Built with cookieless tracking options and native GDPR/CCPA compliance, aiming to give you growth metrics without compromising user privacy.\\n      - superscript\\n      - button \\"View source details for citation from Usermaven. Press Enter to open sources dialog.\\":\\n        - generic: Usermaven\\n  - heading \\"5. Matomo\\" [level=3]\\n  - list:\\n    - paragraph:\\n      - text: \\"Best for:\\"\\n      - text: Teams that want a powerful, direct 1:1 substitute for Google Analytics features (including eCommerce tracking and heatmaps) with absolute data ownership.\\n      - superscript\\n      - button \\"View source details for citation from bootstrap.build. Press Enter to open sources dialog.\\":\\n        - generic: bootstrap.build\\n    - paragraph:\\n      - text: \\"The Tradeoff:\\"\\n      - text: The interface can feel corporate and cluttered—much like GA4—and hosting it yourself requires server maintenance resources.\\n      - superscript\\n      - text: Their cloud version is also noticeably pricier than minimalist alternatives.\\n      - button \\"View source details for citation from bootstrap.build. Press Enter to open sources dialog.\\":\\n        - generic: bootstrap.build\\n    - paragraph:\\n      - text: \\"Privacy Model:\\"\\n      - text: Offers a cookieless tracking mode and complete data ownership (either via self-hosting or secure cloud) so you can avoid consent banners if configured correctly.\\n      - superscript\\n      - button \\"View source details for citation from bootstrap.build. Press Enter to open sources dialog.\\":\\n        - generic: bootstrap.build\\n  - heading \\"Quick Recommendation Summary\\" [level=3]\\n  - list:\\n    - paragraph:\\n      - text: Choose\\n      - generic: Plausible\\n      - text: or\\n      - generic: Simple Analytics\\n      - text: if your SaaS needs a lightweight dashboard just to track marketing site traffic and simple sign-up goals.\\n      - superscript\\n      - button \\"View source details for citation from Usermaven. Press Enter to open sources dialog.\\":\\n        - generic: Usermaven\\n    - paragraph: Choose PostHog if you need deep insight into how users actually use your application inside the product.\\n  - paragraph: Are you leaning more toward a simple traffic dashboard for your marketing site, or do you need deep product analytics to track user behavior inside your SaaS application?\\n","sha256":"e9ea936f25efb05c7be808d0b5d8f36606ad89cdd22442732700d0ff6869afb0","status":"ok"}\n', encoding="utf-8")
subprocess.run([sys.executable,"geo_audit.py","analyze","--config","target.json","--captures","captures.jsonl","--out","audit-output"],check=True)
print(Path("audit-output/audit.md").read_text())
